# Запуск и подключение

Как запустить переносимый Jupyter с плагином, где токен, как подключить MCP-клиентов (Claude Code, локальная модель) и как прогнать тесты.

## Запуск сервера

Jupyter ставится в `jupyter_bin/<win|android>`, его данные и база — в `jupyter_data/` (корень Jupyter; база — `jupyter_data/kb/`, тексты инструментов — `jupyter_data/mcp_docs/`). Каталоги `jupyter_bin/` не анализировать — там только установленные пакеты.

**Windows** — `start_jupyter.bat`. Python по умолчанию `py -3.11`, переопределяется `set PY=путь\к\python.exe`.

**Android (Termux)** — `sh start_jupyter.sh`. Python по умолчанию `python`, переопределяется `PY=python3.12 sh start_jupyter.sh`. Для первой установки один раз: `pkg install python rust clang make libzmq`. Если установлен termux-api, скрипт вызывает `termux-wake-lock`, чтобы процессы не умирали при выключенном экране. JupyterLab открывается в браузере телефона.

**Разделение обязанностей: в скрипте минимум, остальное — в конфиге и плагине.** Скрипт делает только то, что нужно до запуска Python:
1. При первом запуске ставит `jupyterlab` и `dulwich` в `jupyter_bin/<win|android>` (`pip install --target`).
2. Добавляет корень проекта и `jupyter_bin/…` в `PYTHONPATH`; направляет `JUPYTER_CONFIG_DIR`, `JUPYTER_DATA_DIR`, `JUPYTER_RUNTIME_DIR` в `jupyter_data/`.
3. Запускает JupyterLab с корнем `jupyter_data/`. Дополнительные аргументы скрипта передаются Jupyter.

Настройки сервера — в `jupyter_data/config/jupyter_server_config.json`: включённое расширение `kb_mcp`, `ip = 127.0.0.1`, `port = 8888`, браузер не открывается. Токен и вывод адресов — в плагине (см. «Токен и адреса»).

MCP работает только пока запущен Jupyter.

**Переменные окружения плагина:** `KB_MCP_OFF` — отключаемые возможности MCP по умолчанию (см. `development/mcp-kb/api#connection`); `KB_MCP_COMMIT_DELAY` — задержка автокоммита в секундах (по умолчанию 180).

## Токен и адреса

Токен фиксированный, один на веб и MCP. Им занимается плагин (`kb_mcp/launch.py`) при загрузке:
- токен задан явно (`JUPYTER_TOKEN`, `--IdentityProvider.token`, конфиг) — используется он;
- иначе — из `<JUPYTER_CONFIG_DIR>/token.txt` (у нас `jupyter_data/config/token.txt`); нет файла — плагин генерирует токен и сохраняет его туда. Сменить — удалить файл и перезапустить.

После старта плагин печатает в консоль фактические адреса (с реальным портом — если 8888 занят, Jupyter берёт соседний) и готовые к копированию команды подключения к Claude Code с проставленным токеном — по одной на каждый вариант API:
```
[kb] token: <токен>  (jupyter_data/config/token.txt)
[kb] web:   http://127.0.0.1:8888/lab?token=<токен>
[kb] mcp:   http://127.0.0.1:8888/mcp/kb?profile=lite&token=<токен>
[kb] admin: http://127.0.0.1:8888/mcp/kb-admin?token=<токен>
[kb] Claude Code (run in the project folder):
  claude mcp add --transport http -s local kb "http://127.0.0.1:8888/mcp/kb?profile=full&lang=ru&off=structured&token=<токен>"
  claude mcp add --transport http -s local kb "http://127.0.0.1:8888/mcp/kb?profile=std&lang=ru&off=structured&token=<токен>"
  claude mcp add --transport http -s local kb "http://127.0.0.1:8888/mcp/kb?profile=lite&lang=ru&off=structured&token=<токен>"
  claude mcp add --transport http -s local kb-admin "http://127.0.0.1:8888/mcp/kb-admin?lang=ru&off=structured&token=<токен>"
```
Варианты `kb` — альтернативы (одно имя сервера, выбрать профиль под силу модели; для Claude — `full`), `kb-admin` — дополнительно, для индексатора. `-s local` → `-s user`, чтобы сервер был во всех проектах. Уже добавлен — сначала `claude mcp remove kb -s local`.

## Подключение клиентов

Клиенту даётся HTTP-адрес endpoint-а с параметрами (`profile`, `lang`, `limit`, `off`, `token`). Адрес задаёт человек в конфиге клиента; модель его не меняет. Короткая инструкция для нового проекта — `development/mcp-kb/connect`.

- **Claude Code — потребитель:** MCP-сервер `kb` (http, scope local) с `off=structured` — иначе модель видит JSON вместо плоского текста:
  `claude mcp add --transport http -s local kb "http://127.0.0.1:8888/mcp/kb?profile=full&lang=ru&off=structured&token=<токен>"`
- **Claude Code — индексатор:** MCP-сервер `kb-admin` + навык `kb-index` (вызов `/kb-index`, «проиндексируй базу», «наведи порядок»):
  `claude mcp add --transport http -s local kb-admin "http://127.0.0.1:8888/mcp/kb-admin?lang=ru&off=structured&token=<токен>"`
  Текст навыка — статья `development/mcp-kb/skills/kb-index`; установка — сказать агенту «установи навык из …», он соберёт `~/.claude/skills/kb-index/SKILL.md` (см. `#install` той статьи). Границы полномочий индексатор получает от сервера (`mcp_docs/<lang>/admin.ipynb`), навык задаёт порядок работы.
- После смены адреса достаточно переподключить: `/mcp` → сервер → Reconnect.
- **Слабая локальная модель:** `profile=lite` (при необходимости `lang=ru`).
- **Другой индексатор:** `/mcp/kb-admin?token=<токен>` — правила придут в `instructions`.

Агентам-потребителям давать только `/mcp/kb`; админку — человеку или индексатору.

## Тесты

Тесты — `tests/` (`test_core.py`, `test_events.py`, `test_server.py`), pytest на настоящем `ContentsManager` и живом Jupyter Server.

Запуск (Windows, из корня проекта):
```
set PYTHONPATH=C:\projects\ai\android_ai;C:\projects\ai\android_ai\jupyter_bin\win
py -3.11 -m pytest tests -q
```

## Настройки JupyterLab

Пользовательские настройки Lab лежат в проекте: `jupyter_data/config/lab/user-settings/` (переносятся вместе с ним; правятся в Settings → Settings Editor или файлом, после правки файла — обновить страницу).

| Настройка | Файл → ключ | Значение |
|---|---|---|
| Automatically Start Preferred Kernel | `@jupyterlab/notebook-extension/tracker.jupyterlab-settings` → `autoStartDefaultKernel` | `true` |
| Shut Down Kernel on Close | тот же файл → `kernelShutdown` | `true` |
| Autosave Documents | `@jupyterlab/docmanager-extension/plugin.jupyterlab-settings` → `autosave` | `false` (выбор пользователя) |

Зачем именно так и как не получать вопрос «сохранить?» при закрытии нетронутой статьи — `development/mcp-kb/save-prompts`.